# Train the local balancing bot with PPO curriculum learning

This notebook uses `balance_bot_env_modify.py`, `balancing_bot.xml`, and `ppo_trainer_modify.py` from this folder. The robot resets grounded on its wheels with a small pitch-rate disturbance. Phase 1 trains for 500,000 CPU PPO steps with one live MuJoCo viewer and rewards both low pitch and low pitch rate. The MJCF physics timestep is 0.002 seconds and the wheel-joint damping is set in `balancing_bot.xml`.

Evaluate several deterministic episodes and check episode lengths before moving to phase 2. Phase 2 adds penalties for position drift and yaw; those simulated state values would need real sensors or state estimation on hardware. Restart the notebook kernel and run setup cells again before starting a fresh run so the updated environment module is imported.

In [1]:
# Imports
import sys
from pathlib import Path
from dataclasses import replace
import importlib

import gymnasium as gym
import numpy as np
import torch
from gymnasium.wrappers import RecordEpisodeStatistics

# Ensure project root is in sys.path whether executed from root or notebooks/
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import envs
import envs.balance_bot_env as env_module
importlib.reload(env_module)
BalanceBotEnv = env_module.BalanceBotEnv

import rl.ppo as ppo_module
importlib.reload(ppo_module)
from rl.ppo import (
    PPOConfig,
    evaluate,
    export_actor_onnx,
    export_tb_plots_as_csv,
    train,
)


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
# Settings for phase-1 learning
SEED = 42
NUM_ENVS = 4  # Match the original curriculum notebook: more parallel envs stabilizes PPO updates
STEPS_PER_ENV = 500_000
SIM_TIMESTEP = 0.002  # MuJoCo physics integration step in seconds
INITIAL_PITCH_RATE_RANGE = 0.08  # Small recovery disturbance in rad/s

In [3]:
# Configure CPU PPO with a long horizon for control at the 2 ms physics rate
ppo_config = PPOConfig(
    exp_name="balancing-bot-curriculum",
    env_id="BalancingBot-v0",
    seed=SEED,
    cuda=False,
    num_envs=NUM_ENVS,
    actor_hidden_layers=2,
    actor_hidden_size=16,
    critic_hidden_layers=2,
    critic_hidden_size=16,
    total_timesteps=NUM_ENVS * STEPS_PER_ENV,
    num_steps=2048,
    num_minibatches=32,
    update_epochs=10,
    anneal_lr=True,
    learning_rate=3e-4,
    gamma=0.99,
    gae_lambda=0.95,
    clip_coef=0.2,
    value_clip=1.0,
    ent_coef=0.0,
    vf_coef=0.5,
    max_grad_norm=0.5,
    checkpoint_interval=50,
    save_model=True,
    timestep=SIM_TIMESTEP,
)

assert ppo_config.total_timesteps == NUM_ENVS * STEPS_PER_ENV
# Matches the original curriculum notebook: train() computes num_iterations with floor
# division, so total_timesteps doesn't need to divide batch_size (num_envs * num_steps) evenly.

In [4]:
def make_balance_bot_env(render=False, **kwargs):
    """Create a local balance-bot environment, optionally with a MuJoCo viewer."""
    env = BalanceBotEnv(
        render_mode="human" if render else None,
        **kwargs,
    )
    return RecordEpisodeStatistics(env)


def make_envs(num_envs, **kwargs):
    """Render only the first environment to keep CPU training manageable."""
    return gym.vector.SyncVectorEnv(
        [
            lambda render=(index == 0), env_kwargs=kwargs: make_balance_bot_env(
                render=render,
                **env_kwargs,
            )
            for index in range(num_envs)
        ]
    )

In [5]:
# Phase 1: prioritize staying upright; the first environment opens the live viewer.
envs = make_envs(
    NUM_ENVS,
    initial_pitch_rate_range=INITIAL_PITCH_RATE_RANGE,
    pitch_penalty_coef=5.0,
    pitch_rate_penalty_coef=0.2,
    action_penalty_coef=0.01,
    position_penalty_coef=0.0,
    yaw_penalty_coef=0.0,
)

phase1_result = train(ppo_config, envs=envs)
phase1_run_path = phase1_result.checkpoint_dir

# Continue from the best phase-1 checkpoint when available.
phase1_model_path = phase1_result.best_model_path or phase1_result.final_model_path
if phase1_model_path is not None:
    phase1_result.agent.load_state_dict(
        torch.load(phase1_model_path, map_location="cpu", weights_only=True)
    )

result = phase1_result

Run name: BalancingBot-v0__balancing-bot-curriculum__42__1793325010
TensorBoard: http://localhost:6006/#scalars&regexFilter=balancing-bot-curriculum
Checkpoint saved to runs\BalancingBot-v0__balancing-bot-curriculum__42__1793325010\checkpoint_iter0050.cleanrl_model
New best model saved (mean_return=-113.05) to runs\BalancingBot-v0__balancing-bot-curriculum__42__1793325010\best_model.cleanrl_model
Checkpoint saved to runs\BalancingBot-v0__balancing-bot-curriculum__42__1793325010\checkpoint_iter0100.cleanrl_model
New best model saved (mean_return=-88.73) to runs\BalancingBot-v0__balancing-bot-curriculum__42__1793325010\best_model.cleanrl_model
Checkpoint saved to runs\BalancingBot-v0__balancing-bot-curriculum__42__1793325010\checkpoint_iter0150.cleanrl_model
Checkpoint saved to runs\BalancingBot-v0__balancing-bot-curriculum__42__1793325010\checkpoint_iter0200.cleanrl_model
New best model saved (mean_return=21.39) to runs\BalancingBot-v0__balancing-bot-curriculum__42__1793325010\best_mode

In [6]:
# Inspect phase-1 results
print(f"Best model: {phase1_result.best_model_path}")
print(f"Final model: {phase1_result.final_model_path}")
print(f"Best mean return: {phase1_result.best_mean_return:.2f}")

Best model: runs\BalancingBot-v0__balancing-bot-curriculum__42__1793325010\best_model.cleanrl_model
Final model: runs\BalancingBot-v0__balancing-bot-curriculum__42__1793325010\balancing-bot-curriculum_final.cleanrl_model
Best mean return: 21.39


In [11]:
# Evaluate phase 1 in the live viewer at the MuJoCo timestep
eval_config = replace(ppo_config, timestep=0.002)
returns = evaluate(
    phase1_result.agent,
    eval_episodes=3,
    config=eval_config,
    envs=envs,
)
print(f"Phase-1 mean return: {np.nanmean(returns):.2f}")

Phase-1 mean return: 108.90


In [8]:
# Export the phase-1 actor to ONNX for testing or deployment
phase1_model_to_export = (
    phase1_result.best_model_path or phase1_result.final_model_path
)
if phase1_model_to_export is None:
    raise FileNotFoundError("No phase-1 model checkpoint is available for ONNX export")

phase1_obs_size = envs.single_observation_space.shape[0]
phase1_action_size = envs.single_action_space.shape[0]
export_actor_onnx(
    model_path=phase1_model_to_export,
    output_path=phase1_run_path / "actor.onnx",
    obs_size=phase1_obs_size,
    action_size=phase1_action_size,
    num_hidden_layers=ppo_config.actor_hidden_layers,
    hidden_layer_size=ppo_config.actor_hidden_size,
)

d:\RL\3D_CAD\balancing_bot_urdf\urdf\ppo_trainer_modify.py:395: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W1008 07:09:34.536000 10916 Lib\site-packages\torch\onnx\_internal\exporter\_registration.py:107] torchvision is not installed. Skipping torchvision::nms
W1008 07:09:34.547000 10916 Lib\site-packages\torch\onnx\_internal\exporter\_registration.py:107] torchvision is not installed. Skipping torchvision::roi_align
W1008 07:09:34.550000 10916 Lib\site-packages\torch\onnx\_internal\exporter\_registration.py:107] torchvision is not installed. Skipping torchvision::roi_pool
W1008 07:09:34.553000 10916 Lib\site-packages\torch\onnx\_internal\exporter\_registration.py:107] torchvision is not installed. Skipping torchvision::deform_conv2d


[torch.onnx] Obtain model graph for `Sequential([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `Sequential([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...
[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
Actor exported to ONNX: runs\BalancingBot-v0__balancing-bot-curriculum__42__1791341596\actor.onnx


In [27]:
# Export phase-1 TensorBoard metrics to CSV
export_tb_plots_as_csv(phase1_run_path)

Exported charts_metrics.csv (5 metrics, 1779 steps)
Exported losses_metrics.csv (7 metrics, 500 steps)


In [12]:
# Phase 2: add position and yaw penalties on the same running environments, matching the
# original curriculum notebook's in-place coefficient update (no env teardown/rebuild).
for env_stat_wrapper in envs.envs:
    env = env_stat_wrapper.env
    env.position_penalty_coef = 0.001
    env.yaw_penalty_coef = 0.1

result = train(ppo_config, envs=envs, agent=phase1_result.agent)
phase2_run_path = result.checkpoint_dir

Run name: BalancingBot-v0__balancing-bot-curriculum__42__1793327202
TensorBoard: http://localhost:6006/#scalars&regexFilter=balancing-bot-curriculum
Checkpoint saved to runs\BalancingBot-v0__balancing-bot-curriculum__42__1793327202\checkpoint_iter0050.cleanrl_model
New best model saved (mean_return=-375.36) to runs\BalancingBot-v0__balancing-bot-curriculum__42__1793327202\best_model.cleanrl_model
Checkpoint saved to runs\BalancingBot-v0__balancing-bot-curriculum__42__1793327202\checkpoint_iter0100.cleanrl_model
New best model saved (mean_return=-248.22) to runs\BalancingBot-v0__balancing-bot-curriculum__42__1793327202\best_model.cleanrl_model
Checkpoint saved to runs\BalancingBot-v0__balancing-bot-curriculum__42__1793327202\checkpoint_iter0150.cleanrl_model
New best model saved (mean_return=-104.80) to runs\BalancingBot-v0__balancing-bot-curriculum__42__1793327202\best_model.cleanrl_model
Checkpoint saved to runs\BalancingBot-v0__balancing-bot-curriculum__42__1793327202\checkpoint_iter

In [13]:
# Inspect phase-2 results
print(f"Best model: {result.best_model_path}")
print(f"Final model: {result.final_model_path}")
print(f"Best mean return: {result.best_mean_return:.2f}")

Best model: runs\BalancingBot-v0__balancing-bot-curriculum__42__1793327202\best_model.cleanrl_model
Final model: runs\BalancingBot-v0__balancing-bot-curriculum__42__1793327202\balancing-bot-curriculum_final.cleanrl_model
Best mean return: -104.80


In [26]:
# Evaluate phase 2 in the live viewer at the MuJoCo timestep
eval_config = replace(ppo_config, timestep=0.002)
returns = evaluate(
    result.agent,
    eval_episodes=3,
    config=eval_config,
    envs=envs,
)
print(f"Phase-2 mean return: {np.nanmean(returns):.2f}")

Phase-2 mean return: 806.96


In [39]:
# Export phase-2 TensorBoard metrics to CSV
export_tb_plots_as_csv(phase2_run_path)

Export error: OpenBLAS error: Memory allocation still failed after 10 retries, giving up.
OpenBLAS error: Memory allocation still failed after 10 retries, giving up.


In [40]:
# Close the environments
for idx, env in enumerate(envs.envs):
    print(f"Closing env {idx}")
    env.close()

Closing env 0


In [27]:
# Export the phase-2 actor to ONNX for deployment
model_to_export = result.best_model_path or result.final_model_path
if model_to_export is None:
    raise FileNotFoundError("No trained model checkpoint is available for ONNX export")

obs_size = envs.single_observation_space.shape[0]
action_size = envs.single_action_space.shape[0]
export_actor_onnx(
    model_path=model_to_export,
    output_path=result.checkpoint_dir / "actor.onnx",
    obs_size=obs_size,
    action_size=action_size,
    num_hidden_layers=ppo_config.actor_hidden_layers,
    hidden_layer_size=ppo_config.actor_hidden_size,
)

d:\RL\3D_CAD\balancing_bot_urdf\urdf\ppo_trainer_modify.py:395: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W1004 17:46:57.557000 15952 Lib\site-packages\torch\onnx\_internal\exporter\_registration.py:107] torchvision is not installed. Skipping torchvision::nms
W1004 17:46:57.559000 15952 Lib\site-packages\torch\onnx\_internal\exporter\_registration.py:107] torchvision is not installed. Skipping torchvision::roi_align
W1004 17:46:57.560000 15952 Lib\site-packages\torch\onnx\_internal\exporter\_registration.py:107] torchvision is not installed. Skipping torchvision::roi_pool
W1004 17:46:57.561000 15952 Lib\site-packages\torch\onnx\_internal\exporter\_registration.py:107] torchvision is not installed. Skipping torchvision::deform_conv2d


[torch.onnx] Obtain model graph for `Sequential([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `Sequential([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...
[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
Actor exported to ONNX: runs\BalancingBot-v0__balancing-bot-curriculum__42__1791105316\actor.onnx
